In [1]:
!pip install -q -U "ultralytics>=8.3.0" "datasets>=2.19.0" "huggingface_hub>=0.23.0" "pyarrow>=15.0.0" onnx onnxsim pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 82.6 MB/s eta 0:00:00


In [2]:
import os
import io
import json
import math
import random
import shutil
import zipfile
from pathlib import Path

import yaml
import numpy as np
import torch
from PIL import Image
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# =========================
# Основные настройки
# =========================
SEED = 42
HF_DATASET_ID = "nilsleh/dotaV2_patched"

# Выбор runtime:
# "auto" | "gpu" | "cpu" | "tpu"
RUNTIME_TARGET = "auto"

# TPU для Ultralytics OBB в Colab нестабилен.
# Если True — только детектируем TPU и показываем статус.
# Обучение всё равно безопасно пойдёт на CPU, чтобы notebook не падал.
ALLOW_EXPERIMENTAL_TPU = False

# Гиперпараметры
IMG_SIZE = 1024
EPOCHS = 50
BATCH = 8
WORKERS = 2
PATIENCE = 20
VAL_RATIO_IF_MISSING = 0.1
SKIP_DIFFICULT = False

# Пути Colab
ROOT = Path("/content/dota_obb_colab")
YOLO_DATA_DIR = ROOT / "yolo_dataset"
RUNS_DIR = ROOT / "runs"
EXPORT_DIR = ROOT / "export"
ROOT.mkdir(parents=True, exist_ok=True)
YOLO_DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
def detect_runtime(target="auto"):
    target = str(target).lower().strip()

    # GPU
    if target in ("auto", "gpu"):
        if torch.cuda.is_available():
            return {
                "kind": "gpu",
                "train_device": "0",
                "info": torch.cuda.get_device_name(0)
            }
        if target == "gpu":
            print("GPU запрошен, но CUDA недоступна. Будет fallback.")

    # TPU
    if target in ("auto", "tpu"):
        try:
            os.environ.setdefault("PJRT_DEVICE", "TPU")
            import torch_xla.core.xla_model as xm
            xla_device = xm.xla_device()
            return {
                "kind": "tpu",
                "train_device": "cpu",  # безопасный fallback для Ultralytics train()
                "info": str(xla_device)
            }
        except Exception as e:
            if target == "tpu":
                print(f"TPU запрошен, но недоступен: {e}")

    # CPU
    return {
        "kind": "cpu",
        "train_device": "cpu",
        "info": "CPU"
    }

runtime = detect_runtime(RUNTIME_TARGET)
print("Runtime detected:", runtime)

if runtime["kind"] == "tpu":
    print("\n[ВАЖНО]")
    print("TPU обнаружен, но Ultralytics OBB training в Colab TPU работает нестабильно.")
    print("Для надёжности обучение будет выполнено на CPU, если не писать отдельный XLA loop.")
    if ALLOW_EXPERIMENTAL_TPU:
        print("ALLOW_EXPERIMENTAL_TPU=True, но стандартный model.train() всё равно не гарантирует TPU execution.")

TRAIN_DEVICE = runtime["train_device"]

# Автоподстройка параметров под CPU
if TRAIN_DEVICE == "cpu":
    IMG_SIZE = min(IMG_SIZE, 640)
    BATCH = min(BATCH, 2)
    WORKERS = 2
    print(f"\nCPU mode: IMG_SIZE={IMG_SIZE}, BATCH={BATCH}, WORKERS={WORKERS}")
else:
    print(f"\nTraining device: {TRAIN_DEVICE}, IMG_SIZE={IMG_SIZE}, BATCH={BATCH}, WORKERS={WORKERS}")

Runtime detected: {'kind': 'gpu', 'train_device': '0', 'info': 'Tesla T4'}

Training device: 0, IMG_SIZE=1024, BATCH=8, WORKERS=2


In [4]:
WEIGHTS_PATH = hf_hub_download(
    repo_id="Ultralytics/YOLO26",
    filename="yolo26s-obb.pt",
    repo_type="model"
)

print("WEIGHTS_PATH =", WEIGHTS_PATH)
print("Exists:", os.path.exists(WEIGHTS_PATH))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


yolo26s-obb.pt:   0%|          | 0.00/21.7M [00:00<?, ?B/s]

WEIGHTS_PATH = /root/.cache/huggingface/hub/models--Ultralytics--YOLO26/snapshots/de6601a0e3575f8968a22a383021d1099d990857/yolo26s-obb.pt
Exists: True


In [5]:
DOTA_KNOWN_ORDER = [
    "plane",
    "baseball-diamond",
    "bridge",
    "ground-track-field",
    "small-vehicle",
    "large-vehicle",
    "ship",
    "tennis-court",
    "basketball-court",
    "storage-tank",
    "soccer-ball-field",
    "roundabout",
    "harbor",
    "swimming-pool",
    "helicopter",
    "container-crane",
    "airport",
    "helipad",
]

def safe_stem(value, idx):
    if value is None:
        return f"{idx:08d}"
    value = Path(str(value)).stem
    value = "".join(c if c.isalnum() or c in "-_." else "_" for c in value)
    return value or f"{idx:08d}"

def detect_columns(ds):
    cols = list(ds.column_names)
    features = ds.features

    image_col = None
    ann_col = None
    id_col = None

    for c in cols:
        feat = features.get(c, None)
        if feat is not None and feat.__class__.__name__.lower() == "image":
            image_col = c
            break

    if image_col is None:
        for c in ["image", "img", "images"]:
            if c in cols:
                image_col = c
                break

    for c in ["annotations", "annotation", "objects", "labels", "label", "obb", "target", "instances"]:
        if c in cols:
            ann_col = c
            break

    if ann_col is None:
        sample = ds[0]
        for c in cols:
            if c == image_col:
                continue
            v = sample[c]
            if isinstance(v, (dict, list, str)):
                ann_col = c
                break

    for c in ["file_name", "filename", "image_id", "id", "name", "key", "__key__"]:
        if c in cols:
            id_col = c
            break

    if image_col is None or ann_col is None:
        raise RuntimeError(
            f"Не удалось определить image/annotation columns. "
            f"Найденные колонки: {cols}"
        )

    return image_col, ann_col, id_col

def ensure_pil_image(img):
    if isinstance(img, Image.Image):
        return img.convert("RGB")

    if isinstance(img, dict):
        if img.get("bytes") is not None:
            return Image.open(io.BytesIO(img["bytes"])).convert("RGB")
        if img.get("path") is not None:
            return Image.open(img["path"]).convert("RGB")

    if isinstance(img, str):
        return Image.open(img).convert("RGB")

    raise TypeError(f"Неподдерживаемый тип изображения: {type(img)}")

def to_python_obj(x):
    if x is None:
        return {}
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return {}
    return x

def extract_boxes_labels_difficults(annotation):
    ann = to_python_obj(annotation)

    boxes, labels, difficults = [], [], []

    if isinstance(ann, dict):
        if any(k in ann for k in ["boxes", "class_names", "classes", "difficults"]):
            boxes = ann.get("boxes") or ann.get("bboxes") or ann.get("bbox") or ann.get("polygons") or []
            labels = ann.get("class_names")
            if labels is None:
                labels = ann.get("labels")
            if labels is None:
                labels = ann.get("classes")
            difficults = ann.get("difficults") or ann.get("difficult") or [0] * len(boxes)
            return boxes, labels or [], difficults or [0] * len(boxes)

        if "objects" in ann and isinstance(ann["objects"], list):
            for obj in ann["objects"]:
                if not isinstance(obj, dict):
                    continue
                box = obj.get("box") or obj.get("bbox") or obj.get("points") or obj.get("polygon")
                label = obj.get("class_name") or obj.get("label") or obj.get("name") or obj.get("category") or obj.get("class")
                diff = obj.get("difficult", 0)
                if box is not None and label is not None:
                    boxes.append(box)
                    labels.append(label)
                    difficults.append(diff)
            return boxes, labels, difficults

    if isinstance(ann, list):
        for obj in ann:
            if not isinstance(obj, dict):
                continue
            box = obj.get("box") or obj.get("bbox") or obj.get("points") or obj.get("polygon")
            label = obj.get("class_name") or obj.get("label") or obj.get("name") or obj.get("category") or obj.get("class")
            diff = obj.get("difficult", 0)
            if box is not None and label is not None:
                boxes.append(box)
                labels.append(label)
                difficults.append(diff)
        return boxes, labels, difficults

    return [], [], []

def order_points_clockwise(pts):
    pts = np.asarray(pts, dtype=np.float32).reshape(-1, 2)
    center = pts.mean(axis=0)
    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
    pts = pts[np.argsort(angles)]
    start = np.argmin(pts.sum(axis=1))
    pts = np.roll(pts, -start, axis=0)
    return pts

def box_to_4pts(box):
    arr = np.asarray(box, dtype=np.float32)

    if arr.ndim == 1 and arr.size == 8:
        pts = arr.reshape(4, 2)
        return order_points_clockwise(pts)

    if arr.ndim == 2 and arr.shape == (4, 2):
        return order_points_clockwise(arr)

    if arr.ndim == 1 and arr.size == 4:
        x1, y1, x2, y2 = arr.tolist()
        pts = np.array([
            [x1, y1],
            [x2, y1],
            [x2, y2],
            [x1, y2]
        ], dtype=np.float32)
        return order_points_clockwise(pts)

    if arr.ndim == 2 and arr.shape == (2, 4):
        pts = arr.T
        return order_points_clockwise(pts)

    flat = arr.reshape(-1)
    if flat.size >= 8:
        pts = flat[:8].reshape(4, 2)
        return order_points_clockwise(pts)

    raise ValueError(f"Не удалось интерпретировать box формата: shape={arr.shape}")

def normalize_obb_points(box, width, height):
    pts = box_to_4pts(box).copy()
    pts[:, 0] = np.clip(pts[:, 0] / max(width, 1), 0.0, 1.0)
    pts[:, 1] = np.clip(pts[:, 1] / max(height, 1), 0.0, 1.0)
    return pts.reshape(-1).tolist()

def choose_train_val_splits(ds_dict, val_ratio=0.1, seed=42):
    split_names = list(ds_dict.keys())
    print("Найдены split'ы:", split_names)

    if "train" in ds_dict:
        train_ds = ds_dict["train"]
        if "validation" in ds_dict:
            val_ds = ds_dict["validation"]
        elif "val" in ds_dict:
            val_ds = ds_dict["val"]
        elif "valid" in ds_dict:
            val_ds = ds_dict["valid"]
        elif "test" in ds_dict:
            val_ds = ds_dict["test"]
        else:
            tmp = train_ds.train_test_split(test_size=val_ratio, seed=seed)
            train_ds, val_ds = tmp["train"], tmp["test"]
        return {"train": train_ds, "val": val_ds}

    first_name = split_names[0]
    base_ds = ds_dict[first_name]
    tmp = base_ds.train_test_split(test_size=val_ratio, seed=seed)
    return {"train": tmp["train"], "val": tmp["test"]}

def collect_class_names(ds_splits):
    names_found = []
    names_set = set()

    for split_name, ds in ds_splits.items():
        _, ann_col, _ = detect_columns(ds)
        for row in ds:
            boxes, labels, _ = extract_boxes_labels_difficults(row[ann_col])
            for lab in labels:
                if lab is None:
                    continue
                lab = str(lab).strip()
                if lab and lab not in names_set and not lab.isdigit():
                    names_set.add(lab)
                    names_found.append(lab)

    if names_found:
        if set(names_found).issubset(set(DOTA_KNOWN_ORDER)):
            names_found = [x for x in DOTA_KNOWN_ORDER if x in names_set]
        else:
            names_found = sorted(names_found)
        return names_found

    raise RuntimeError("Не удалось собрать class_names из датасета.")

def write_yolo_obb_split(ds, split_name, class_to_id, out_root, skip_difficult=False):
    image_col, ann_col, id_col = detect_columns(ds)

    img_dir = out_root / "images" / split_name
    lbl_dir = out_root / "labels" / split_name
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    total_images = 0
    total_labels = 0
    empty_labels = 0
    skipped_boxes = 0

    for idx, row in enumerate(ds):
        img = ensure_pil_image(row[image_col])
        w, h = img.size

        raw_id = row[id_col] if id_col and id_col in row else idx
        stem = f"{split_name}_{safe_stem(raw_id, idx)}"

        img_path = img_dir / f"{stem}.png"
        lbl_path = lbl_dir / f"{stem}.txt"

        img.save(img_path, format="PNG")

        boxes, labels, difficults = extract_boxes_labels_difficults(row[ann_col])
        lines = []

        for i, box in enumerate(boxes):
            label = labels[i] if i < len(labels) else None
            difficult = difficults[i] if i < len(difficults) else 0

            if label is None:
                skipped_boxes += 1
                continue

            if skip_difficult:
                try:
                    if int(difficult) != 0:
                        continue
                except Exception:
                    pass

            label_str = str(label).strip()

            try:
                if label_str in class_to_id:
                    cls_id = class_to_id[label_str]
                elif label_str.isdigit():
                    cls_id = int(label_str)
                else:
                    skipped_boxes += 1
                    continue

                coords = normalize_obb_points(box, w, h)
                if len(coords) != 8:
                    skipped_boxes += 1
                    continue

                line = f"{cls_id} " + " ".join(f"{v:.6f}" for v in coords)
                lines.append(line)
            except Exception:
                skipped_boxes += 1
                continue

        lbl_path.write_text("\n".join(lines), encoding="utf-8")

        total_images += 1
        total_labels += len(lines)
        if len(lines) == 0:
            empty_labels += 1

    print(
        f"[{split_name}] images={total_images}, labels={total_labels}, "
        f"empty_label_files={empty_labels}, skipped_boxes={skipped_boxes}"
    )

In [6]:
dataset = load_dataset(HF_DATASET_ID)
print(dataset)

for split_name in dataset.keys():
    print("\nSPLIT:", split_name)
    print("Columns:", dataset[split_name].column_names)
    print("Features:", dataset[split_name].features)

README.md:   0%|          | 0.00/849 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

train/train-000000.tar:   0%|          | 0.00/321M [00:00<?, ?B/s]

train/train-000001.tar:   0%|          | 0.00/322M [00:00<?, ?B/s]

train/train-000002.tar:   0%|          | 0.00/319M [00:00<?, ?B/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/22 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['png', 'json', '__key__', '__url__'],
        num_rows: 34277
    })
    validation: Dataset({
        features: ['png', 'json', '__key__', '__url__'],
        num_rows: 5108
    })
    test: Dataset({
        features: ['png', 'json', '__key__', '__url__'],
        num_rows: 11698
    })
})

SPLIT: train
Columns: ['png', 'json', '__key__', '__url__']
Features: {'png': Image(mode=None, decode=True), 'json': {'boxes': List(List(Value('float64'))), 'classes': List(Value('int64')), 'class_names': List(Value('string')), 'difficults': List(Value('int64')), 'metadata': {}, 'original_filename': Value('string')}, '__key__': Value('string'), '__url__': Value('string')}

SPLIT: validation
Columns: ['png', 'json', '__key__', '__url__']
Features: {'png': Image(mode=None, decode=True), 'json': {'boxes': List(List(Value('float64'))), 'classes': List(Value('int64')), 'class_names': List(Value('string')), 'difficults': List(Value('int64')), 'metada

In [7]:
splits = choose_train_val_splits(dataset, val_ratio=VAL_RATIO_IF_MISSING, seed=SEED)

print("Train size:", len(splits["train"]))
print("Val size:", len(splits["val"]))

class_names = collect_class_names(splits)
class_to_id = {name: i for i, name in enumerate(class_names)}

print("\nClasses:")
for i, name in enumerate(class_names):
    print(i, "->", name)

Найдены split'ы: ['train', 'validation', 'test']
Train size: 34277
Val size: 5108

Classes:
0 -> plane
1 -> baseball-diamond
2 -> bridge
3 -> ground-track-field
4 -> small-vehicle
5 -> large-vehicle
6 -> ship
7 -> tennis-court
8 -> basketball-court
9 -> storage-tank
10 -> soccer-ball-field
11 -> roundabout
12 -> harbor
13 -> swimming-pool
14 -> helicopter
15 -> container-crane
16 -> airport
17 -> helipad


In [8]:
if YOLO_DATA_DIR.exists():
    shutil.rmtree(YOLO_DATA_DIR)
YOLO_DATA_DIR.mkdir(parents=True, exist_ok=True)

write_yolo_obb_split(
    splits["train"],
    split_name="train",
    class_to_id=class_to_id,
    out_root=YOLO_DATA_DIR,
    skip_difficult=SKIP_DIFFICULT
)

write_yolo_obb_split(
    splits["val"],
    split_name="val",
    class_to_id=class_to_id,
    out_root=YOLO_DATA_DIR,
    skip_difficult=SKIP_DIFFICULT
)

data_yaml = {
    "path": str(YOLO_DATA_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": class_names,
}

DATA_YAML_PATH = ROOT / "dota_obb.yaml"
with open(DATA_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print("DATA_YAML_PATH =", DATA_YAML_PATH)
print(DATA_YAML_PATH.read_text(encoding="utf-8"))

[train] images=34277, labels=439376, empty_label_files=1419, skipped_boxes=0
[val] images=5108, labels=71898, empty_label_files=211, skipped_boxes=0
DATA_YAML_PATH = /content/dota_obb_colab/dota_obb.yaml
path: /content/dota_obb_colab/yolo_dataset
train: images/train
val: images/val
names:
- plane
- baseball-diamond
- bridge
- ground-track-field
- small-vehicle
- large-vehicle
- ship
- tennis-court
- basketball-court
- storage-tank
- soccer-ball-field
- roundabout
- harbor
- swimming-pool
- helicopter
- container-crane
- airport
- helipad



In [9]:
train_imgs = sorted((YOLO_DATA_DIR / "images" / "train").glob("*.png"))
train_lbls = sorted((YOLO_DATA_DIR / "labels" / "train").glob("*.txt"))
val_imgs = sorted((YOLO_DATA_DIR / "images" / "val").glob("*.png"))
val_lbls = sorted((YOLO_DATA_DIR / "labels" / "val").glob("*.txt"))

print("train images:", len(train_imgs))
print("train labels:", len(train_lbls))
print("val images:", len(val_imgs))
print("val labels:", len(val_lbls))

if train_lbls:
    print("\nПример label-файла:")
    print(train_lbls[0])
    print(train_lbls[0].read_text(encoding="utf-8")[:1000])

train images: 34277
train labels: 34277
val images: 5108
val labels: 5108

Пример label-файла:
/content/dota_obb_colab/yolo_dataset/labels/train/train_000000.txt
2 0.431641 0.658203 0.505859 0.652344 0.492188 0.853516 0.425781 0.859375


In [ ]:
model = YOLO(WEIGHTS_PATH)

train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    workers=WORKERS,
    device=TRAIN_DEVICE,   # "0" для GPU, "cpu" для CPU/TPU fallback
    project=str(RUNS_DIR),
    name="yolo26s_obb_colab_finetune",
    patience=PATIENCE,
    optimizer="AdamW",
    lr0=1e-3,
    lrf=1e-2,
    weight_decay=5e-4,
    warmup_epochs=3.0,
    cos_lr=True,
    amp=(TRAIN_DEVICE != "cpu"),
    cache=False,
    seed=SEED,
    deterministic=True,
    verbose=True,
)

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dota_obb_colab/dota_obb.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/.cache/huggingface/hub/models--Ultralytics--YOLO26/snapshots/de6601a0e3575f8968a22a383021d1099d990857/yolo26s-obb.pt, momentum=0.937, mosaic=1.0, multi_s

In [ ]:
train_dir = Path(model.trainer.save_dir)
best_pt = train_dir / "weights" / "best.pt"
last_pt = train_dir / "weights" / "last.pt"

best_weights = best_pt if best_pt.exists() else last_pt
print("Best weights:", best_weights)

best_model = YOLO(str(best_weights))

metrics = best_model.val(
    data=str(DATA_YAML_PATH),
    split="val",
    imgsz=IMG_SIZE,
    batch=max(1, BATCH // 2),
    device=TRAIN_DEVICE
)

print(metrics)

In [ ]:
use_half = (TRAIN_DEVICE != "cpu")

onnx_path = best_model.export(
    format="onnx",
    imgsz=IMG_SIZE,
    opset=13,
    simplify=True,
    half=use_half,
    dynamic=False
)

onnx_path = Path(onnx_path)
print("ONNX path:", onnx_path)
print("ONNX size MB:", round(onnx_path.stat().st_size / 1024 / 1024, 2))

In [ ]:
FINAL_EXPORT_DIR = EXPORT_DIR / "final_bundle"
if FINAL_EXPORT_DIR.exists():
    shutil.rmtree(FINAL_EXPORT_DIR)
FINAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1) Копируем ONNX
final_onnx_path = FINAL_EXPORT_DIR / onnx_path.name
shutil.copy2(onnx_path, final_onnx_path)

# 2) Копируем best.pt
if best_weights.exists():
    shutil.copy2(best_weights, FINAL_EXPORT_DIR / best_weights.name)

# 3) Копируем весь run directory целиком
RUN_COPY_DIR = FINAL_EXPORT_DIR / "runs_export"
shutil.copytree(train_dir, RUN_COPY_DIR, dirs_exist_ok=True)

print("Содержимое FINAL_EXPORT_DIR:")
for p in FINAL_EXPORT_DIR.rglob("*"):
    print(p)

In [ ]:
archive_path = EXPORT_DIR / "yolo26s_obb_colab_export.zip"

if archive_path.exists():
    archive_path.unlink()

with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in FINAL_EXPORT_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(FINAL_EXPORT_DIR))

print("Archive created:", archive_path)
print("Archive size MB:", round(archive_path.stat().st_size / 1024 / 1024, 2))

In [ ]:
from google.colab import files

files.download(str(archive_path))